# Predicting Hospital Readmission in Diabetic Patients
## Notebook 1 of 7 - Data Cleaning & Preprocessing

In this notebook I cleaned, imputed, encoded, and scaled the raw `diabetic_data.csv` file. The cleaned, scaled outputs were saved as `.npy` arrays and passed forward to all modeling notebooks.

**Steps I followed:**
1. Loaded the raw data
2. Identified and handled missing values (MCAR drop, special-value imputation, mode imputation)
3. Transformed features (age ordinal mapping, ICD-9 grouping, one-hot encoding)
4. Performed a 70/30 train/test split and applied StandardScaler normalization
5. Saved all outputs for use in downstream notebooks

## 1. Import Libraries

In [12]:
import numpy as np
import pandas as pd

## 2. Load Raw Data

In [13]:
diabetic = pd.read_csv('diabetic_data.csv')
df = diabetic.copy()  # I made a copy so I never accidentally modified the original
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 3. Identifying Missing Data

The raw dataset encoded missing values as `'?'` rather than `NaN`. I replaced them first, then looped through every column to count how many values were missing and what percentage that represented before deciding on an imputation strategy.

**Decisions I made by column:**
- `weight` (97% missing), `payer_code` (40%), `medical_specialty` (49%) → **dropped** (MCAR, not worth imputing for analysis and no statistical significance)
- `race`, `max_glu_serum`, `A1Cresult` → **special-value imputation** (undocumented ≠ truly missing)
- `diag_1`, `diag_2`, `diag_3` (<2% missing each) → **mode imputation**

In [14]:
df = df.replace('?', np.nan)  # replaced all missing data labeled '?' with NaN

for col in df.columns:
    count = df[col].isna().sum()
    percentage_check = (count / len(df)) * 100
    print(f"{col}: {count} missing ({percentage_check:.2f}%)")

encounter_id: 0 missing (0.00%)
patient_nbr: 0 missing (0.00%)
race: 2273 missing (2.23%)
gender: 0 missing (0.00%)
age: 0 missing (0.00%)
weight: 98569 missing (96.86%)
admission_type_id: 0 missing (0.00%)
discharge_disposition_id: 0 missing (0.00%)
admission_source_id: 0 missing (0.00%)
time_in_hospital: 0 missing (0.00%)
payer_code: 40256 missing (39.56%)
medical_specialty: 49949 missing (49.08%)
num_lab_procedures: 0 missing (0.00%)
num_procedures: 0 missing (0.00%)
num_medications: 0 missing (0.00%)
number_outpatient: 0 missing (0.00%)
number_emergency: 0 missing (0.00%)
number_inpatient: 0 missing (0.00%)
diag_1: 21 missing (0.02%)
diag_2: 358 missing (0.35%)
diag_3: 1423 missing (1.40%)
number_diagnoses: 0 missing (0.00%)
max_glu_serum: 96420 missing (94.75%)
A1Cresult: 84748 missing (83.28%)
metformin: 0 missing (0.00%)
repaglinide: 0 missing (0.00%)
nateglinide: 0 missing (0.00%)
chlorpropamide: 0 missing (0.00%)
glimepiride: 0 missing (0.00%)
acetohexamide: 0 missing (0.00%)


### 3a. Drop High-Missingness Columns (MCAR)

In [15]:
# I removed weight, payer_code, and medical_specialty — all MCAR with no statistical significance
df = df.drop(columns=['weight', 'payer_code', 'medical_specialty'])
print(f"Columns after dropping high-missingness features: {df.shape[1]}")

Columns after dropping high-missingness features: 47


### 3b. Special-Value Imputation (MAR)

For `race`, `max_glu_serum`, and `A1Cresult`, I used special-value imputation rather than a statistical method. These fields were MAR - their absence carried clinical meaning. I filled them with descriptive labels to preserve that context rather than hiding it. For `max_glu_serum` and `A1Cresult` specifically, a blank field didn't mean the result was missing - it meant the test simply wasn't ordered, which the Strack et al. paper confirmed happened in only 18.4% of encounters.

In [16]:
df['race'] = df['race'].fillna('Not recorded')                   # documentation gap, not truly absent
df['max_glu_serum'] = df['max_glu_serum'].fillna('Not tested')   # test was not ordered
df['A1Cresult'] = df['A1Cresult'].fillna('Not tested')           # only ordered in 18.4% of encounters

### 3c. Mode Imputation for Diagnosis Fields

For the three diagnosis fields, which each had less than 2% missing, I used mode imputation. The values were sparse enough that anything more complex like KNN would have added unnecessary work for minimal benefit.

In [17]:
for col in ["diag_1", "diag_2", "diag_3"]:
    mode_value = df[col].mode()[0]
    missing_count = df[col].isna().sum()
    df[col] = df[col].fillna(mode_value)
    print(f"{col}: imputed {missing_count} values with mode '{mode_value}'")

diag_1: imputed 21 values with mode '428'
diag_2: imputed 358 values with mode '276'
diag_3: imputed 1423 values with mode '250'


## 4. Feature Transformations

### 4a. Age - Ordinal Mapping

Age was stored as string intervals in the raw data (e.g., `'[60-70)'`). Because age has a real ordinal structure, I mapped the ten intervals to integers 0 through 9 rather than one-hot encoding them. One-hot encoding would have treated each age group as a completely separate and unrelated category, which would have discarded the ordering that actually mattered here.

In [18]:
age_mapping = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, '[40-50)': 4,
    '[50-60)': 5, '[60-70)': 6, '[70-80)': 7, '[80-90)': 8, '[90-100)': 9
}
df['age'] = df['age'].map(age_mapping)
print("Age value counts after mapping:")
print(df['age'].value_counts().sort_index())

Age value counts after mapping:
age
0      161
1      691
2     1657
3     3775
4     9685
5    17256
6    22483
7    26068
8    17197
9     2793
Name: count, dtype: int64


### 4b. Diagnosis Codes - ICD-9 Grouping

Each of the three diagnosis fields contained over 700 unique ICD-9-CM codes, which was too many categories to model directly. I wrote a grouping function that collapsed everything into eight clinically meaningful disease categories. E and V codes were sent to `'Other'` - E codes describe how an injury happened rather than the injury itself, and V codes describe healthcare contacts for non-illness reasons like a routine checkup or medication refill, making both supplementary rather than primary diagnoses.

In [19]:
def icd_codes_grouping(code):
    try:
        num = float(str(code))
        if 250 <= num < 251:    return 'Diabetes'
        elif 140 <= num <= 239: return 'Neoplasms'
        elif 390 <= num <= 459: return 'Circulatory'
        elif 460 <= num <= 519: return 'Respiratory'
        elif 520 <= num <= 579: return 'Digestive'
        elif 580 <= num <= 629: return 'Genitourinary'
        elif 710 <= num <= 739: return 'Musculoskeletal'
        elif 800 <= num <= 999: return 'Injury'
        else: return 'Other'
    except:
        return 'Other'  # handles E/V codes that start with a letter

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(icd_codes_grouping)

print("diag_1 category counts:")
print(df['diag_1'].value_counts())

diag_1 category counts:
diag_1
Circulatory        30357
Other              22595
Respiratory        10407
Digestive           9208
Diabetes            8757
Injury              6974
Genitourinary       5078
Musculoskeletal     4957
Neoplasms           3433
Name: count, dtype: int64


### 4c. Label Encode Target Variable (y)

I used label encoding to map the three outcomes to integers: `NO → 0`, `>30 → 1`, `<30 → 2`. One thing worth noting is that mapping these outcomes to 0, 1, and 2 technically implied an order that doesn't really exist - being readmitted within 30 days is not "more" than being readmitted after 30 days in any meaningful way. However, this is a standard approach for multiclass problems, and the models I used don't treat these numbers as a scale.

In [20]:
df['readmitted'] = df['readmitted'].map({'NO': 0, '>30': 1, '<30': 2})
print("Target class distribution:")
print(df['readmitted'].value_counts())

Target class distribution:
readmitted
0    54864
1    35545
2    11357
Name: count, dtype: int64


### 4d. Binary Encode Gender

In [21]:
# I used simple binary encoding since 'Unknown/Invalid' affected only 3 rows (1 = Male, 0 = Female)
df['gender'] = (df['gender'] == 'Male').astype(int)
print("Gender value counts:", df['gender'].value_counts().to_dict())

Gender value counts: {0: 54711, 1: 47055}


### 4e. One-Hot Encode Remaining Categorical Variables

I one-hot encoded all remaining string columns with `drop_first=True` to prevent perfect multicollinearity. This encoding process was what expanded the feature count from the original 47 columns to 98 - each unique category in a variable became a new binary column, and dropping one redundant column per feature prevented the dummy variable trap.

In [22]:
categorical_cols = df.select_dtypes(include='object').columns.tolist() #'include='str' had to be changed for jupyter functionality 
print(f"Categorical columns to encode: {categorical_cols}")
print(f"Total categorical features: {len(categorical_cols)}")

df_onehot = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Converted boolean output to int for compatibility with the modeling pipeline
boolean_output = df_onehot.select_dtypes(include='bool').columns
df_onehot[boolean_output] = df_onehot[boolean_output].astype(int)

# Moved readmitted (y) back to the end since one-hot encoding had displaced it
cols = [col for col in df_onehot.columns if col != 'readmitted'] + ['readmitted']
df_onehot = df_onehot[cols]

print(f"\nShape after one-hot encoding: {df_onehot.shape}")
df_onehot.head()

Categorical columns to encode: ['race', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']
Total categorical features: 31

Shape after one-hot encoding: (101766, 101)


,encounter_id,patient_nbr,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,...,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-rosiglitazone_Steady,metformin-pioglitazone_Steady,change_No,diabetesMed_Yes,readmitted
0,2278392,8222157,0,0,6,25,1,1,41,0,...,1,0,0,0,0,0,0,1,0,0
1,149190,55629189,0,1,1,1,7,3,59,0,...,1,0,0,0,0,0,0,0,1,1
2,64410,86047875,0,2,1,1,7,2,11,5,...,1,0,0,0,0,0,0,1,1,0
3,500364,82442376,1,3,1,1,7,2,44,1,...,1,0,0,0,0,0,0,0,1,0
4,16680,42519267,1,4,1,1,7,1,51,0,...,1,0,0,0,0,0,0,0,1,0


### 4f. Save Cleaned Dataset

In [23]:
df_onehot.to_excel('diabetic_data_cleaned.xlsx', index=False)
print("Saved cleaned dataset to diabetic_data_cleaned.xlsx")

Saved cleaned dataset to diabetic_data_cleaned.xlsx


## 5. Train/Test Split and Feature Scaling

I used a 70/30 split, then applied `StandardScaler` - fitted **only on the training set** - to both train and test. If I had fit the scaler on the full dataset before splitting, it would have learned statistics from the test set and used them during training, which would have resulted in data leakage and made the model look better than it actually was on data it had never seen.

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# I dropped encounter_id and patient_nbr since they serve no analytic value
X = df_onehot.drop(columns=['encounter_id', 'patient_nbr', 'readmitted'])
y = df_onehot['readmitted']

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

Feature matrix shape: (101766, 98)
Target shape: (101766,)


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Train size: {X_train.shape[0]:,} rows")
print(f"Test size:  {X_test.shape[0]:,} rows")

Train size: 71,236 rows
Test size:  30,530 rows


In [26]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on training data to learn mean/SD per column
X_test_scaled  = scaler.transform(X_test)        # applied the same learned transform to the test set

print("Scaling complete.")
print(f"X_train_scaled mean (should ≈ 0): {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std  (should ≈ 1): {X_train_scaled.std():.6f}")

Scaling complete.
X_train_scaled mean (should ≈ 0): 0.000000
X_train_scaled std  (should ≈ 1): 1.000000


## 6. Save Arrays for Downstream Notebooks

In [27]:
np.save('X_train_scaled.npy', X_train_scaled)
np.save('X_test_scaled.npy',  X_test_scaled)
np.save('y_train.npy', y_train)
np.save('y_test.npy',  y_test)

print("Saved:")
print("  X_train_scaled.npy", X_train_scaled.shape)
print("  X_test_scaled.npy ", X_test_scaled.shape)
print("  y_train.npy        ", y_train.shape)
print("  y_test.npy         ", y_test.shape)

Saved:
  X_train_scaled.npy (71236, 98)
  X_test_scaled.npy  (30530, 98)
  y_train.npy         (71236,)
  y_test.npy          (30530,)
